# Solution: Debug Drill 16 - CNN Overfitting

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
# Load data
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train_full[:2000].astype('float32') / 255.0
y_train = y_train_full[:2000]
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## Bug Identification

The buggy model has **severe overfitting** due to:

1. **Too many parameters for the data size**
   - ~1.5M parameters for only 2000 training samples
   - Ratio of ~750 parameters per sample (should be <10)

2. **No regularization**
   - No Dropout layers
   - No BatchNormalization
   - No early stopping

3. **Too deep architecture**
   - 5 conv layers + 2 dense layers for simple 28x28 images
   - Overkill for Fashion-MNIST

## The Fix

Apply multiple regularization techniques:

In [ ]:
# FIXED MODEL with proper regularization

fixed_model = keras.Sequential([
    # Simpler architecture
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

fixed_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Fixed model parameters: {fixed_model.count_params():,}")
print(f"Params per sample: {fixed_model.count_params() / len(X_train):.1f}")

In [ ]:
# Train with early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Training fixed model...")
history = fixed_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Verify fix
train_loss, train_acc = fixed_model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = fixed_model.evaluate(X_test, y_test, verbose=0)

print(f"\n=== Fixed Model Results ===")
print(f"Train Accuracy: {train_acc:.1%}")
print(f"Test Accuracy: {test_acc:.1%}")
print(f"Gap: {(train_acc - test_acc)*100:.1f}%")

gap = train_acc - test_acc
if gap < 0.15:
    print("\n✅ Overfitting significantly reduced!")

## Postmortem

- **Root cause:** The model had 1.5M parameters for only 2000 training samples (750 params/sample), with no regularization. This allowed the model to memorize training data instead of learning generalizable patterns.

- **Fix applied:** Simplified architecture (32-64 filters vs 64-128-256), added Dropout (0.25 after conv, 0.5 after dense), added BatchNormalization, and used early stopping. Reduced parameters to ~50K.

- **Prevention:** Always check the params-to-samples ratio (keep under 10:1 for small datasets). Use regularization by default. Monitor train-test gap during development. Consider data augmentation for image tasks with limited data.